[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/708san/CV_data/blob/main/notebooks/cv/classification_v1.ipynb)

# CV教材：画像を種類に分ける

## はじめに

このNotebookでは、画像分類の課題を作り、ResNet分類モデルを学習して結果を確かめます。
使うデータは、出典と固定版を確認したトマトの葉の画像です。画像の判定は植物の病気の確定診断ではありません。


## 1. 自分の課題を決める

画像から何を分けたいか、誰の役に立つか、成功したといえる条件を書きます。

- 困りごと：
- 分けたい種類：
- 使う人：
- 成功したといえる条件：
- 試す入力と期待する出力：

## 2. データの出典と利用条件を確認する

このNotebookでは、次の固定版だけを使います。

- データ：トマトの葉 5クラス
- 取得ID：`tomato_leaves` または `トマトの葉`
- 固定版：`tomato-v1.2.0`
- 元データの表記：CC0 1.0
- 注意：画像の判定は医療・農業上の確定診断ではありません。

元画像をNotebookへ再収録したり、ラベルを書き換えたりしません。

In [ ]:
# ColabではT4 GPUを選んでから、このセルを実行します。
%pip install "cv-data-lesson[torch]==0.1.0"

from cvlesson import datasets
from cvlesson.models import ResNetClassifier, available_devices
import pandas as pd

print("利用できる装置:", available_devices())
data = datasets.get("tomato_leaves")
print("出典・利用条件:", data.source_info)
print("分割:", data.splits)
frame = data.to_df()
frame.head()


## 3. 画像とラベルを確認する

`label_ja`は日本語で表示するための名前、`label_id`は学習で使う番号です。
`test`は最後の評価用なので、学習には使いません。

In [ ]:
print(frame[['image_path', 'label', 'label_ja', 'label_id', 'split', 'sample_id']].head())
print(frame['label_ja'].value_counts())
data.show_examples(9, split='train')

## 4. 小さな分類モデルを動かす

ResNet18の画像特徴を使い、最後の出力層を学習します。まずは各クラスから決まった枚数だけ選び、実データで短く試します。trainで学習し、validationで途中の傾向を確認します。testは最後の確認まで使いません。


In [ ]:
import torch

# GPUが使える環境では明示して利用し、なければCPUで実行します。
device = "cuda" if available_devices()["cuda"] else "cpu"
print("学習装置:", device)

def クラスごとに選ぶ(rows, 枚数, seed=42):
    available = int(rows["label"].value_counts().min())
    return (
        rows.groupby("label", group_keys=False)
        .sample(n=min(枚数, available), random_state=seed)
        .reset_index(drop=True)
    )

train_all = frame[frame["split"] == "train"]
val_all = frame[frame["split"] == "val"]
test_all = frame[frame["split"] == "test"]
train_data = クラスごとに選ぶ(train_all, 100)
val_data = クラスごとに選ぶ(val_all, 25)
test_data = クラスごとに選ぶ(test_all, 40)
print("学習/検証/テストの件数:", len(train_data), len(val_data), len(test_data))

model = ResNetClassifier(
    device=device,
    epochs=3,
    batch_size=16,
    image_size=128,
    random_state=42,
)
model.fit(train_data["image_path"], train_data["label"])
print("学習損失:", [round(value, 4) for value in model.loss_history_])
print("validation正解率:", round(model.score(val_data["image_path"], val_data["label"]), 3))
print("分類ラベル:", model.classes_)


## 5. 見本を試す

学習に使っていないtest画像で正解率を確認し、予測ラベルとクラスごとの確率目安を表示します。結果はこの小さな学習設定での測定値で、実際の植物を診断する精度を表しません。


In [ ]:
from IPython.display import display
from PIL import Image

print("test正解率:", round(model.score(test_data["image_path"], test_data["label"]), 3))

label_ja = dict(zip(data.classes, data.class_names_ja))
predicted = model.predict(test_data["image_path"])
probabilities = model.predict_proba(test_data["image_path"])
test_result = test_data[["image_path", "label", "label_ja"]].copy()
test_result["予測"] = predicted
test_result["予測_日本語"] = [label_ja[label] for label in predicted]
test_result["予測確率の目安"] = probabilities.max(axis=1).round(3)
display(test_result[["label_ja", "予測_日本語", "予測確率の目安"]].head(10))

def 予測する(index):
    row = test_data.iloc[index]
    with Image.open(row["image_path"]) as opened:
        display(opened.convert("RGB"))
    probability = model.predict_proba([row["image_path"]])[0]
    predicted_label = model.classes_[int(probability.argmax())]
    return {
        "予測": label_ja[predicted_label],
        "正解": row["label_ja"],
        "予測確率の目安": float(probability.max()),
    }

print(予測する(0))


## 6. 自分の課題に合わせて試す

次の欄に、試した番号、期待した結果、実際の結果を書きます。モデルが間違えたときは、画像の見え方や種類の似ている点も記録します。

- 試した入力：
- 期待した結果：
- 実際の結果：
- 次に変えたいこと：

In [ ]:
# test画像の番号を変えて、別の未見画像も予測します。
試す番号 = 1
予測する(試す番号)


## 7. 振り返り

- うまく分類できた種類：
- 間違えやすかった種類：
- test画像で確かめたこと：
- データの制約や、まだ分からないこと：
- 次に改善するなら：